# **Aula: EDA — Censo Escolar 2022**
## Link: https://www.kaggle.com/datasets/hugobovaretohorsth/censo-escolar-2022-brazil/data

# Análise Exploratória do Censo Escolar 2022

## Contexto da base

Nesta aula, utilizaremos uma base derivada do Censo Escolar 2022, com informações sobre escolas brasileiras, incluindo localização, dependência administrativa, Unidade Federativa, quantidade de matrículas, quantidade de docentes e indicadores de infraestrutura escolar.

A unidade de análise deste estudo é a escola. Portanto, as análises realizadas ao longo do notebook têm como foco entender diferenças estruturais entre escolas, e não o desempenho individual de alunos.

## Objetivo da análise

O objetivo desta análise é investigar desigualdades de infraestrutura entre escolas brasileiras e construir uma narrativa analítica capaz de apoiar decisões de priorização.

A partir da análise exploratória, buscamos responder perguntas como:

- Como as escolas estão distribuídas entre os estados brasileiros?
- Qual é a diferença entre escolas urbanas e rurais?
- Quais itens de infraestrutura estão mais presentes ou mais ausentes?
- Onde estão os maiores sinais de vulnerabilidade estrutural?
- Quais características ajudam a identificar escolas com maior vulnerabilidade?

## Dor de negócio

Imagine uma secretaria de educação, uma fundação ou uma área pública responsável por apoiar investimentos em infraestrutura escolar.

O desafio é que existem muitas escolas, diferentes realidades regionais e orçamento limitado. Nesse cenário, não basta dizer que “as escolas precisam de investimento”. É necessário identificar onde estão os maiores gaps, quais grupos são mais vulneráveis e como priorizar ações com base em evidências.

A análise exploratória entra justamente nesse ponto: transformar uma base grande e técnica em informação útil para decisão.

Neste notebook, vamos usar visualizações com Plotly, construção de indicadores e uma modelagem simples para apoiar a seguinte pergunta central:

**Quais escolas apresentam maior vulnerabilidade de infraestrutura e quais características ajudam a explicar essa desigualdade?**

In [ ]:
import os

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
import scipy.sparse as sp
import shap

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

shap.initjs()

In [ ]:
# Paleta inspirada na identidade visual da FIAP
FIAP_CINZA_ESCURO = "#1F1F1F"
FIAP_CINZA_MEDIO = "#6E6E6E"
FIAP_CINZA_CLARO = "#F4F4F4"
FIAP_BRANCO = "#FFFFFF"
FIAP_ROSA = "#ED145B"
FIAP_ROSA_CLARO = "#FDE6EE"
FIAP_ROSA_ESCURO = "#4A001C"
FIAP_AZUL_ESCURO = "#071D49"
FIAP_ROXO = "#6A00FF"
FIAP_AZUL = "#00A3FF"

fiap_template = go.layout.Template()

fiap_template.layout = go.Layout(
    font=dict(family="Arial, sans-serif", size=13, color=FIAP_CINZA_ESCURO),
    title=dict(font=dict(family="Arial, sans-serif", size=22, color=FIAP_AZUL_ESCURO), x=0.02),
    paper_bgcolor=FIAP_BRANCO,
    plot_bgcolor=FIAP_CINZA_CLARO,
    colorway=[FIAP_ROSA, FIAP_AZUL_ESCURO, FIAP_ROXO, FIAP_AZUL, FIAP_CINZA_MEDIO],
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        linecolor=FIAP_CINZA_MEDIO,
        tickfont=dict(color=FIAP_CINZA_ESCURO),
        # titlefont=dict(color=FIAP_CINZA_ESCURO),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#DDDDDD",
        zeroline=False,
        linecolor=FIAP_CINZA_MEDIO,
        tickfont=dict(color=FIAP_CINZA_ESCURO),
        # titlefont=dict(color=FIAP_CINZA_ESCURO),
    ),
    legend=dict(
        title_font=dict(color=FIAP_CINZA_ESCURO),
        font=dict(color=FIAP_CINZA_ESCURO),
        bgcolor="rgba(255,255,255,0)",
    ),
)

pio.templates["fiap"] = fiap_template
pio.templates.default = "fiap"


def aplicar_layout_fiap(
    fig, altura=500, margem_esquerda=80, margem_direita=40, margem_superior=80, margem_inferior=40
):
    """
    Aplica padronização visual inspirada na identidade da FIAP.
    Permite controlar margens para evitar corte de rótulos.
    """
    fig.update_layout(
        template="fiap",
        title_x=0.02,
        height=altura,
        margin=dict(l=margem_esquerda, r=margem_direita, t=margem_superior, b=margem_inferior),
    )

    fig.update_yaxes(automargin=True)

    fig.update_xaxes(automargin=True)

    return fig

In [ ]:
# !pip install kagglehub pandas numpy plotly scikit-learn nbformat


import plotly.graph_objects as go

# Download da base pelo KaggleHub
path = kagglehub.dataset_download("hugobovaretohorsth/censo-escolar-2022-brazil")

print("Path to dataset files:", path)

# Encontrando arquivos CSV
csv_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("Arquivos CSV encontrados:")
for arquivo in csv_files:
    print(arquivo)

# Selecionando o primeiro CSV encontrado
csv_path = csv_files[0]

# Lendo a base
try:
    df = pd.read_csv(csv_path, sep=";", encoding="latin1", low_memory=False)
except Exception:
    df = pd.read_csv(csv_path, low_memory=False)

df.head()

In [ ]:
nomes_infra = {
    "IN_AGUA_POTAVEL": "Água Potável",
    "IN_ENERGIA_REDE_PUBLICA": "Energia Rede Pública",
    "IN_ESGOTO_REDE_PUBLICA": "Esgoto Rede Pública",
    "IN_INTERNET": "Internet",
    "IN_BANDA_LARGA": "Banda Larga",
    "IN_LABORATORIO_INFORMATICA": "Laboratório Informática",
    "IN_BIBLIOTECA": "Biblioteca",
    "IN_QUADRA_ESPORTES": "Quadra Esportes",
}

nomes_variaveis_grafico = {
    "SG_UF": "UF",
    "dependencia": "Dependência administrativa",
    "localizacao": "Localização",
    "QT_MAT_BAS": "Quantidade de matrículas",
    "QT_DOC_BAS": "Quantidade de docentes",
    "accuracy": "Acurácia",
    "precision_classe_1": "Precisão — Alta vulnerabilidade",
    "recall_classe_1": "Recall — Alta vulnerabilidade",
    "f1_classe_1": "F1-score — Alta vulnerabilidade",
    "auc": "AUC",
}

### 1. Diagnóstico inicial

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.sample(5).T

In [ ]:
nulos = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum().values,
        "perc_nulos": (df.isna().mean().values * 100).round(2),
    }
).sort_values("perc_nulos", ascending=False)

nulos.head(30)

### 1.1 Dicionários de variáveis

In [ ]:
dicionario_variaveis = pd.DataFrame(
    {
        "variavel": [
            "CO_ENTIDADE",
            "NO_ENTIDADE",
            "SG_UF",
            "NO_MUNICIPIO",
            "TP_DEPENDENCIA",
            "TP_LOCALIZACAO",
            "TP_SITUACAO_FUNCIONAMENTO",
            "IN_AGUA_POTAVEL",
            "IN_ENERGIA_REDE_PUBLICA",
            "IN_ESGOTO_REDE_PUBLICA",
            "IN_INTERNET",
            "IN_BANDA_LARGA",
            "IN_LABORATORIO_INFORMATICA",
            "IN_BIBLIOTECA",
            "IN_QUADRA_ESPORTES",
            "IN_ACESSIBILIDADE_INEXISTENTE",
            "QT_MAT_BAS",
            "QT_DOC_BAS",
        ],
        "descricao": [
            "Código único da escola",
            "Nome da escola",
            "Sigla da Unidade Federativa",
            "Nome do município",
            "Tipo de dependência administrativa da escola",
            "Localização da escola, urbana ou rural",
            "Situação de funcionamento da escola",
            "Indica se a escola possui água potável",
            "Indica se a escola possui energia da rede pública",
            "Indica se a escola possui esgoto ligado à rede pública",
            "Indica se a escola possui acesso à internet",
            "Indica se a escola possui internet banda larga",
            "Indica se a escola possui laboratório de informática",
            "Indica se a escola possui biblioteca",
            "Indica se a escola possui quadra de esportes",
            "Indica ausência de recursos de acessibilidade",
            "Quantidade de matrículas na educação básica",
            "Quantidade de docentes na educação básica",
        ],
        "tipo_conceitual": [
            "Identificador",
            "Identificador",
            "Geográfica",
            "Geográfica",
            "Categórica",
            "Categórica",
            "Categórica",
            "Binária",
            "Binária",
            "Binária",
            "Binária",
            "Binária",
            "Binária",
            "Binária",
            "Binária",
            "Binária invertida",
            "Numérica",
            "Numérica",
        ],
        "observacao": [
            "Não deve ser usada como variável explicativa no modelo",
            "Usada apenas para identificação e consulta",
            "Pode ser usada para análise territorial",
            "Pode ser usada para análises municipais",
            "1 = Federal, 2 = Estadual, 3 = Municipal, 4 = Privada",
            "1 = Urbana, 2 = Rural",
            "Pode ser usada para filtrar escolas em funcionamento",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Normalmente 1 = Sim, 0 = Não",
            "Atenção: 1 indica inexistência de acessibilidade",
            "Pode indicar porte da escola",
            "Pode indicar porte e estrutura da escola",
        ],
    }
)

dicionario_variaveis["existe_na_base"] = dicionario_variaveis["variavel"].isin(df.columns)

dicionario_variaveis

### 2. Seleção de variáveis

In [ ]:
colunas_interesse = [
    "CO_ENTIDADE",
    "NO_ENTIDADE",
    "SG_UF",
    "NO_MUNICIPIO",
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",
    "TP_SITUACAO_FUNCIONAMENTO",
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_INTERNET",
    "IN_BANDA_LARGA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_QUADRA_ESPORTES",
    "IN_ACESSIBILIDADE_INEXISTENTE",
    "QT_MAT_BAS",
    "QT_DOC_BAS",
]

colunas_existentes = [c for c in colunas_interesse if c in df.columns]

df_eda = df[colunas_existentes].copy()

df_eda.head()

### 2.1 Padronizando categorias

In [ ]:
map_dependencia = {1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada"}

map_localizacao = {1: "Urbana", 2: "Rural"}

if "TP_DEPENDENCIA" in df_eda.columns:
    df_eda["dependencia"] = df_eda["TP_DEPENDENCIA"].map(map_dependencia)

if "TP_LOCALIZACAO" in df_eda.columns:
    df_eda["localizacao"] = df_eda["TP_LOCALIZACAO"].map(map_localizacao)

df_eda[["TP_DEPENDENCIA", "dependencia", "TP_LOCALIZACAO", "localizacao"]].head()

### 3. Gráficos com Plotly

### 3.1 Escolas por UF

In [ ]:
escolas_uf = (
    df_eda.groupby("SG_UF", as_index=False)
    .size()
    .rename(columns={"size": "qtd_escolas"})
    .sort_values("qtd_escolas", ascending=False)
)

fig = px.bar(
    escolas_uf,
    x="SG_UF",
    y="qtd_escolas",
    text="qtd_escolas",
    title="Quantidade de escolas por UF — Censo Escolar 2022",
    labels={"SG_UF": "UF", "qtd_escolas": "Quantidade de escolas"},
)

fig.update_traces(marker_color=FIAP_ROSA, textposition="outside")

fig.update_layout(yaxis_title=None, xaxis_title=None)

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

# 3.2 Escolas por localização

In [ ]:
loc = df_eda.groupby("localizacao", as_index=False).size().rename(columns={"size": "qtd_escolas"})

fig = px.pie(
    loc,
    names="localizacao",
    values="qtd_escolas",
    hole=0.55,
    title="Distribuição das escolas por localização",
    color="localizacao",
    color_discrete_map={"Urbana": FIAP_ROSA, "Rural": FIAP_AZUL_ESCURO},
)

fig.update_traces(textinfo="percent+label", textfont_size=14)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

In [ ]:
loc_bar = loc.copy()

loc_bar["percentual"] = loc_bar["qtd_escolas"] / loc_bar["qtd_escolas"].sum() * 100

fig = px.bar(
    loc_bar,
    x="localizacao",
    y="percentual",
    text=loc_bar["percentual"].round(1),
    title="Distribuição das escolas por localização",
    labels={"localizacao": "Localização", "percentual": "% de escolas"},
    color="localizacao",
    color_discrete_map={"Urbana": FIAP_ROSA, "Rural": FIAP_AZUL_ESCURO},
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")

fig.update_layout(yaxis_range=[0, 100], showlegend=False)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

### 3.3 Indicadores de infraestrutura

In [ ]:
infra_cols = [
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_INTERNET",
    "IN_BANDA_LARGA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_QUADRA_ESPORTES",
]

infra_cols = [col for col in infra_cols if col in df_eda.columns]

infra_summary = []

for col in infra_cols:
    infra_summary.append(
        {"indicador": nomes_infra.get(col, col), "perc_escolas_com_item": df_eda[col].mean() * 100}
    )

infra_summary = pd.DataFrame(infra_summary)

infra_summary = infra_summary.sort_values("perc_escolas_com_item", ascending=True)

fig = px.bar(
    infra_summary,
    x="perc_escolas_com_item",
    y="indicador",
    orientation="h",
    text=infra_summary["perc_escolas_com_item"].round(1),
    title="Percentual de escolas com cada item de infraestrutura",
    labels={"perc_escolas_com_item": "% de escolas", "indicador": ""},
)

fig.update_traces(marker_color=FIAP_ROSA, texttemplate="%{text:.1f}%", textposition="outside")

fig.update_layout(xaxis_range=[0, 105], yaxis_title=None)

fig = aplicar_layout_fiap(fig, altura=550)

fig.show()

### 3.4 Internet por localização

In [ ]:
internet_loc = df_eda.groupby("localizacao", as_index=False)["IN_INTERNET"].mean()

internet_loc["perc_com_internet"] = internet_loc["IN_INTERNET"] * 100

fig = px.bar(
    internet_loc,
    x="localizacao",
    y="perc_com_internet",
    text=internet_loc["perc_com_internet"].round(1),
    title="Acesso à internet por localização da escola",
    labels={"localizacao": "Localização", "perc_com_internet": "% de escolas com internet"},
    color="localizacao",
    color_discrete_map={"Urbana": FIAP_ROSA, "Rural": FIAP_AZUL_ESCURO},
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")

fig.update_layout(yaxis_range=[0, 100], showlegend=False)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

### 3.5 Internet por UF e localização

In [ ]:
internet_uf_loc = df_eda.groupby(["SG_UF", "localizacao"], as_index=False)["IN_INTERNET"].mean()

internet_uf_loc["perc_com_internet"] = internet_uf_loc["IN_INTERNET"] * 100

fig = px.bar(
    internet_uf_loc,
    x="SG_UF",
    y="perc_com_internet",
    color="localizacao",
    barmode="group",
    title="Acesso à internet por UF e localização",
    labels={
        "SG_UF": "UF",
        "perc_com_internet": "% de escolas com internet",
        "localizacao": "Localização",
    },
    color_discrete_map={"Urbana": FIAP_ROSA, "Rural": FIAP_AZUL_ESCURO},
)

fig.update_layout(yaxis_range=[0, 100])

fig = aplicar_layout_fiap(fig, altura=550)

fig.show()

### 4. Criando um score de vulnerabilidade

In [ ]:
itens_infra = [
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_INTERNET",
    "IN_BANDA_LARGA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_QUADRA_ESPORTES",
]

itens_infra = [c for c in itens_infra if c in df_eda.columns]

for col in itens_infra:
    df_eda[col] = df_eda[col].fillna(0)

df_eda["score_vulnerabilidade"] = 0

for col in itens_infra:
    df_eda["score_vulnerabilidade"] += np.where(df_eda[col] == 1, 0, 1)

df_eda[["score_vulnerabilidade"]].describe()

### 4.1 Distribuição do score

In [ ]:
score_dist = (
    df_eda.groupby("score_vulnerabilidade", as_index=False)
    .size()
    .rename(columns={"size": "qtd_escolas"})
)

fig = px.bar(
    score_dist,
    x="score_vulnerabilidade",
    y="qtd_escolas",
    text="qtd_escolas",
    title="Distribuição do score de vulnerabilidade de infraestrutura",
    labels={
        "score_vulnerabilidade": "Score de vulnerabilidade",
        "qtd_escolas": "Quantidade de escolas",
    },
)

fig.update_traces(marker_color=FIAP_ROSA, textposition="outside")

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

### 4.2 Score médio por UF

In [ ]:
score_uf = (
    df_eda.groupby("SG_UF", as_index=False)["score_vulnerabilidade"]
    .mean()
    .sort_values("score_vulnerabilidade", ascending=False)
)

fig = px.bar(
    score_uf,
    x="SG_UF",
    y="score_vulnerabilidade",
    text=score_uf["score_vulnerabilidade"].round(2),
    title="Score médio de vulnerabilidade por UF",
    labels={"SG_UF": "UF", "score_vulnerabilidade": "Score médio"},
)

fig.update_traces(marker_color=FIAP_ROSA, textposition="outside")

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

### 4.3 Score por rede e localização

In [ ]:
score_dep_loc = df_eda.groupby(["dependencia", "localizacao"], as_index=False)[
    "score_vulnerabilidade"
].mean()

fig = px.bar(
    score_dep_loc,
    x="dependencia",
    y="score_vulnerabilidade",
    color="localizacao",
    barmode="group",
    text=score_dep_loc["score_vulnerabilidade"].round(2),
    title="Score médio de vulnerabilidade por rede administrativa e localização",
    labels={
        "dependencia": "Dependência administrativa",
        "score_vulnerabilidade": "Score médio",
        "localizacao": "Localização",
    },
    color_discrete_map={"Urbana": FIAP_ROSA, "Rural": FIAP_AZUL_ESCURO},
)

fig.update_traces(textposition="outside")

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

### 5. Mapa do Brasil com Plotly

In [ ]:
url_geojson = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

geojson = requests.get(url_geojson).json()

score_uf_mapa = score_uf.copy()

fig = px.choropleth(
    score_uf_mapa,
    geojson=geojson,
    locations="SG_UF",
    featureidkey="properties.sigla",
    color="score_vulnerabilidade",
    color_continuous_scale=[[0.0, FIAP_ROSA_CLARO], [0.5, FIAP_ROSA], [1.0, FIAP_ROSA_ESCURO]],
    title="Mapa do score médio de vulnerabilidade por UF",
    labels={"score_vulnerabilidade": "Score médio"},
)

fig.update_geos(fitbounds="locations", visible=False)

fig = aplicar_layout_fiap(fig, altura=650)

fig.show()

### 6. Modelagem simples

In [ ]:
limite_alta_vulnerabilidade = df_eda["score_vulnerabilidade"].quantile(0.75)

df_eda["alta_vulnerabilidade"] = np.where(
    df_eda["score_vulnerabilidade"] >= limite_alta_vulnerabilidade, 1, 0
)

df_eda["alta_vulnerabilidade"].value_counts(normalize=True)

### 6.1 Features simples

In [ ]:
features_categoricas = ["SG_UF", "dependencia", "localizacao"]

features_numericas = ["QT_MAT_BAS", "QT_DOC_BAS"]

features_categoricas = [c for c in features_categoricas if c in df_eda.columns]
features_numericas = [c for c in features_numericas if c in df_eda.columns]

features = features_categoricas + features_numericas

model_df = df_eda[features + ["alta_vulnerabilidade"]].dropna()

X = model_df[features]
y = model_df["alta_vulnerabilidade"]

### 6.2 Treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1995, stratify=y
)

### 6.3 Pipeline

In [ ]:
preprocessador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
        ("num", "passthrough", features_numericas),
    ]
)

modelo = RandomForestClassifier(
    n_estimators=200, random_state=1995, class_weight="balanced", max_depth=8
)

pipeline = Pipeline(steps=[("preprocessador", preprocessador), ("modelo", modelo)])

pipeline.fit(X_train, y_train)

### 6.4 Avaliação: Comparando métricas de treino e teste

In [ ]:
y_train_pred = pipeline.predict(X_train)
y_train_proba = pipeline.predict_proba(X_train)[:, 1]

y_test_pred = pipeline.predict(X_test)
y_test_proba = pipeline.predict_proba(X_test)[:, 1]

metricas_modelo = pd.DataFrame(
    {
        "base": ["Treino", "Teste"],
        "accuracy": [accuracy_score(y_train, y_train_pred), accuracy_score(y_test, y_test_pred)],
        "precision_classe_1": [
            precision_score(y_train, y_train_pred),
            precision_score(y_test, y_test_pred),
        ],
        "recall_classe_1": [
            recall_score(y_train, y_train_pred),
            recall_score(y_test, y_test_pred),
        ],
        "f1_classe_1": [f1_score(y_train, y_train_pred), f1_score(y_test, y_test_pred)],
        "auc": [roc_auc_score(y_train, y_train_proba), roc_auc_score(y_test, y_test_proba)],
    }
)

metricas_modelo

### 6.5 Comparação AUC de treino e teste

In [ ]:
fig = px.bar(
    metricas_modelo,
    x="base",
    y="auc",
    text=metricas_modelo["auc"].round(3),
    title="Comparação da AUC entre treino e teste",
    labels={"base": "Base", "auc": "AUC"},
    color="base",
    color_discrete_map={"Treino": FIAP_ROSA, "Teste": FIAP_AZUL_ESCURO},
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")

fig.update_layout(yaxis_range=[0, 1], showlegend=False)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

### 6.6 Várias métricas de treino e teste

In [ ]:
metricas_long = metricas_modelo.melt(
    id_vars="base",
    value_vars=["accuracy", "precision_classe_1", "recall_classe_1", "f1_classe_1", "auc"],
    var_name="metrica",
    value_name="valor",
)

metricas_long["metrica_tratada"] = (
    metricas_long["metrica"].map(nomes_variaveis_grafico).fillna(metricas_long["metrica"])
)

fig = px.bar(
    metricas_long,
    x="metrica_tratada",
    y="valor",
    color="base",
    barmode="group",
    text=metricas_long["valor"].round(3),
    title="Métricas do modelo — treino versus teste",
    labels={"metrica_tratada": "Métrica", "valor": "Valor", "base": "Base"},
    color_discrete_map={"Treino": FIAP_ROSA, "Teste": FIAP_AZUL_ESCURO},
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")

fig.update_layout(yaxis_range=[0, 1], xaxis_tickangle=-20)

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

### 6.7 Curva ROC/AUC - treino e teste

In [ ]:
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_proba)

auc_train = roc_auc_score(y_train, y_train_proba)

fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_proba)

auc_test = roc_auc_score(y_test, y_test_proba)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=fpr_train,
        y=tpr_train,
        mode="lines",
        name=f"Treino — AUC = {auc_train:.3f}",
        line=dict(color=FIAP_ROSA, width=3),
    )
)

fig.add_trace(
    go.Scatter(
        x=fpr_test,
        y=tpr_test,
        mode="lines",
        name=f"Teste — AUC = {auc_test:.3f}",
        line=dict(color=FIAP_AZUL_ESCURO, width=3),
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Classificador aleatório",
        line=dict(color=FIAP_AZUL_ESCURO, width=3, dash="dash"),
    )
)

fig.update_layout(
    title="Curva ROC — Treino versus teste",
    xaxis_title="Taxa de falsos positivos",
    yaxis_title="Taxa de verdadeiros positivos",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    legend=dict(x=0.50, y=0.05, bgcolor="rgba(255,255,255,0.85)"),
)

fig = aplicar_layout_fiap(fig, altura=600)

fig.show()

### 6.7 Relatório de treino e teste

In [ ]:
print("Métricas no treino")
print(classification_report(y_train, y_train_pred))

print("\nMétricas no teste")
print(classification_report(y_test, y_test_pred))

print("\nAUC treino:", round(roc_auc_score(y_train, y_train_proba), 3))
print("AUC teste:", round(roc_auc_score(y_test, y_test_proba), 3))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100

labels = np.array(
    [
        [f"{cm[0, 0]:,}<br>{cm_percent[0, 0]:.1f}%", f"{cm[0, 1]:,}<br>{cm_percent[0, 1]:.1f}%"],
        [f"{cm[1, 0]:,}<br>{cm_percent[1, 0]:.1f}%", f"{cm[1, 1]:,}<br>{cm_percent[1, 1]:.1f}%"],
    ]
)

fig = px.imshow(
    cm,
    text_auto=False,
    title="Matriz de confusão com percentual por classe real — Alta vulnerabilidade",
    labels=dict(x="Predito", y="Real", color="Quantidade"),
    x=["Não alta", "Alta"],
    y=["Não alta", "Alta"],
    color_continuous_scale=[[0.0, FIAP_ROSA_CLARO], [0.5, FIAP_ROSA], [1.0, FIAP_ROSA_ESCURO]],
)

fig.update_traces(text=labels, texttemplate="%{text}", textfont_size=15)

fig = aplicar_layout_fiap(fig, altura=500)

fig.show()

In [ ]:
resultado_importancia = permutation_importance(
    pipeline, X_test, y_test, n_repeats=5, random_state=1995, scoring="roc_auc"
)

importancia = pd.DataFrame(
    {"variavel": X_test.columns, "importancia": resultado_importancia.importances_mean}
)

importancia["variavel_tratada"] = (
    importancia["variavel"].map(nomes_variaveis_grafico).fillna(importancia["variavel"])
)

importancia = importancia.sort_values("importancia", ascending=True)

fig = px.bar(
    importancia,
    x="importancia",
    y="variavel_tratada",
    orientation="h",
    title="Importância das variáveis para prever alta vulnerabilidade",
    labels={"importancia": "Importância média", "variavel_tratada": ""},
)

fig.update_traces(marker_color=FIAP_ROSA)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

### 7. Interpretando a saída do modelo

### 8. Preparar a base para SHAP

In [ ]:
preprocessador_treinado = pipeline.named_steps["preprocessador"]
modelo_treinado = pipeline.named_steps["modelo"]


X_test_shap = X_test.sample(n=min(2000, len(X_test)), random_state=1995)


X_test_shap_trans = preprocessador_treinado.transform(X_test_shap)


if sp.issparse(X_test_shap_trans):
    X_test_shap_trans = X_test_shap_trans.toarray()


feature_names = preprocessador_treinado.get_feature_names_out()


X_test_shap_df = pd.DataFrame(X_test_shap_trans, columns=feature_names, index=X_test_shap.index)

X_test_shap_df.head()

### 8.1 Calculando valores SHAP

In [ ]:
explainer = shap.TreeExplainer(modelo_treinado)
shap_values = explainer.shap_values(X_test_shap_df)

if isinstance(shap_values, list):
    shap_values_classe_1 = shap_values[1]
else:
    if len(shap_values.shape) == 3:
        shap_values_classe_1 = shap_values[:, :, 1]
    else:
        shap_values_classe_1 = shap_values

shap_values_classe_1.shape

### 8.2 Gráfico SHAP de importância global

In [ ]:
shap.summary_plot(shap_values_classe_1, X_test_shap_df, plot_type="bar", max_display=20)

### 8.3 Gráfico SHAP tipo beeswarm

In [ ]:
shap.summary_plot(shap_values_classe_1, X_test_shap_df, max_display=20)

### 8.4 Criar importância SHAP em DataFrame

In [ ]:
shap_importancia = pd.DataFrame(
    {"variavel": feature_names, "shap_medio_abs": np.abs(shap_values_classe_1).mean(axis=0)}
)

shap_importancia = shap_importancia.sort_values("shap_medio_abs", ascending=False)

shap_importancia.head(20)

### 8.5 Limpar nomes técnicos das variáveis SHAP

In [ ]:
def limpar_nome_shap(nome):
    nome = nome.replace("cat__", "").replace("num__", "")
    nome = nome.replace("SG_UF_", "UF = ")
    nome = nome.replace("dependencia_", "Dependência = ")
    nome = nome.replace("localizacao_", "Localização = ")
    nome = nome.replace("QT_MAT_BAS", "Quantidade de matrículas")
    nome = nome.replace("QT_DOC_BAS", "Quantidade de docentes")
    return nome


shap_importancia["variavel_tratada"] = shap_importancia["variavel"].apply(limpar_nome_shap)

shap_importancia.head(20)

### 8.6 Gráfico SHAP com Plotly

In [ ]:
shap_top20 = shap_importancia.head(20).sort_values("shap_medio_abs", ascending=True)

fig = px.bar(
    shap_top20,
    x="shap_medio_abs",
    y="variavel_tratada",
    orientation="h",
    title="Importância SHAP das variáveis — Alta vulnerabilidade",
    labels={"shap_medio_abs": "Impacto médio absoluto no modelo", "variavel_tratada": ""},
)

fig.update_traces(marker_color=FIAP_ROSA)

fig = aplicar_layout_fiap(fig, altura=650)

fig.show()

### 8.7 Agregar SHAP por variável original

In [ ]:
def agrupar_variavel_original(nome):
    nome_limpo = nome.replace("cat__", "").replace("num__", "")

    if nome_limpo.startswith("SG_UF_"):
        return "UF"
    elif nome_limpo.startswith("dependencia_"):
        return "Dependência administrativa"
    elif nome_limpo.startswith("localizacao_"):
        return "Localização"
    elif nome_limpo == "QT_MAT_BAS":
        return "Quantidade de matrículas"
    elif nome_limpo == "QT_DOC_BAS":
        return "Quantidade de docentes"
    else:
        return nome_limpo


shap_importancia["grupo_variavel"] = shap_importancia["variavel"].apply(agrupar_variavel_original)

shap_grupo = (
    shap_importancia.groupby("grupo_variavel", as_index=False)["shap_medio_abs"]
    .sum()
    .sort_values("shap_medio_abs", ascending=True)
)

shap_grupo

### 8.8 Gráfico SHAP agregado por variável original

In [ ]:
fig = px.bar(
    shap_grupo,
    x="shap_medio_abs",
    y="grupo_variavel",
    orientation="h",
    title="Importância SHAP agregada por variável original",
    labels={"shap_medio_abs": "Impacto médio absoluto agregado", "grupo_variavel": ""},
)

fig.update_traces(marker_color=FIAP_ROSA)

fig = aplicar_layout_fiap(fig, altura=450)

fig.show()

### 8.9 Explicação individual de uma escola

In [ ]:
idx = 0

expected_value = explainer.expected_value

if isinstance(expected_value, list):
    expected_value_classe_1 = expected_value[1]
else:
    if isinstance(expected_value, np.ndarray):
        expected_value_classe_1 = (
            expected_value[1] if len(expected_value) > 1 else expected_value[0]
        )
    else:
        expected_value_classe_1 = expected_value


shap.plots._waterfall.waterfall_legacy(
    expected_value_classe_1, shap_values_classe_1[idx], X_test_shap_df.iloc[idx], max_display=15
)